<a href="https://colab.research.google.com/github/AIML-Dept/Adhisha_Sreedith_1GA23AI002/blob/main/Week_05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
pip install qiskit qiskit-aer scipy numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 85.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 106.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 4.9 MB/s eta 0:00:00


In [3]:
import numpy as np
from scipy.stats import binom, chi2
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator


simulator = AerSimulator()

# **Easy: Measure a Hadamard-superposition qubit with 100, 1000 and 10000 shots and tabulate how the measured probability converges to 0.5.**

In [5]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

simulator = AerSimulator()

for shots in [100, 1000, 10000]:

    qc = QuantumCircuit(1, 1)

    # Create |+> state
    qc.h(0)

    # Measure
    qc.measure(0, 0)

    result = simulator.run(qc, shots=shots).result()
    counts = result.get_counts()

    zeros = counts.get("0", 0)
    ones = counts.get("1", 0)

    print("Shots:", shots)
    print("0:", zeros)
    print("1:", ones)
    print("P(0):", zeros / shots)
    print("P(1):", ones / shots)
    print()


Shots: 100
0: 56
1: 44
P(0): 0.56
P(1): 0.44

Shots: 1000
0: 514
1: 486
P(0): 0.514
P(1): 0.486

Shots: 10000
0: 5044
1: 4956
P(0): 0.5044
P(1): 0.4956



# **Medium: Measure a qubit in the X-basis (apply H before measurement) after preparing |+> and |-> states and interpret the results.**

In [6]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

simulator = AerSimulator()
shots = 1000

# Prepare |+>
plus = QuantumCircuit(1, 1)
plus.h(0)       # |0> -> |+>
plus.h(0)       # X-basis measurement
plus.measure(0, 0)

# Prepare |->
minus = QuantumCircuit(1, 1)
minus.x(0)      # |0> -> |1>
minus.h(0)      # |1> -> |->
minus.h(0)      # X-basis measurement
minus.measure(0, 0)

# Run |+>
result_plus = simulator.run(
    plus,
    shots=shots
).result()

# Run |->
result_minus = simulator.run(
    minus,
    shots=shots
).result()

print("|+> state:")
print(result_plus.get_counts())

print("\n|-> state:")
print(result_minus.get_counts())


|+> state:
{'0': 1000}

|-> state:
{'1': 1000}


# **Hard: Perform partial measurement on one qubit of a 2-qubit Bell pair and analyse the resulting state of the unmeasured qubit.**

In [7]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

simulator = AerSimulator()

shots = 1000

# Create Bell state
qc = QuantumCircuit(2, 1)

qc.h(0)
qc.cx(0, 1)

# Measure only qubit 0
qc.measure(0, 0)

result = simulator.run(
    qc,
    shots=shots
).result()

counts = result.get_counts()

print("Bell pair measurement:")
print(counts)

print("\nInterpretation:")
print("If qubit 0 = 0, qubit 1 becomes |0>.")
print("If qubit 0 = 1, qubit 1 becomes |1>.")


Bell pair measurement:
{'0': 514, '1': 486}

Interpretation:
If qubit 0 = 0, qubit 1 becomes |0>.
If qubit 0 = 1, qubit 1 becomes |1>.


# **Real-world: Use repeated measurement statistics to estimate an unknown rotation angle applied to a qubit (simple quantum state tomography).**

In [8]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

simulator = AerSimulator()

# Unknown rotation angle
true_theta = np.pi / 3       # 60 degrees
shots = 10000

qc = QuantumCircuit(1, 1)

# Apply unknown Ry rotation
qc.ry(true_theta, 0)

# Measure
qc.measure(0, 0)

result = simulator.run(
    qc,
    shots=shots
).result()

counts = result.get_counts()

ones = counts.get("1", 0)
p1 = ones / shots

# Estimate theta:
# P(1) = sin^2(theta/2)
estimated_theta = 2 * np.arcsin(np.sqrt(p1))

print("Measurement counts:", counts)
print("Estimated P(1):", p1)

print(
    "True angle:",
    np.degrees(true_theta),
    "degrees"
)

print(
    "Estimated angle:",
    np.degrees(estimated_theta),
    "degrees"
)


Measurement counts: {'1': 2463, '0': 7537}
Estimated P(1): 0.2463
True angle: 59.99999999999999 degrees
Estimated angle: 59.50920029077929 degrees


# **Challenge: Design an experiment that distinguishes a genuinely random quantum bit generator from a biased classical pseudo-random generator using statistical hypothesis testing on measurement data.**

In [9]:
import numpy as np
from scipy.stats import norm
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

simulator = AerSimulator()


# ------------------------------------------------------------
# Generate quantum random bits
# ------------------------------------------------------------

def quantum_random_bits(shots):

    qc = QuantumCircuit(1, 1)

    # |0> -> |+>
    qc.h(0)

    # Measure
    qc.measure(0, 0)

    result = simulator.run(
        qc,
        shots=shots
    ).result()

    counts = result.get_counts()

    bits = (
        [0] * counts.get("0", 0)
        + [1] * counts.get("1", 0)
    )

    return np.array(bits)


# ------------------------------------------------------------
# Generate intentionally biased classical bits
# ------------------------------------------------------------

def classical_random_bits(shots):

    return np.random.binomial(
        1,
        0.51,
        shots
    )


# ------------------------------------------------------------
# Statistical hypothesis test
# H0: p = 0.5
# H1: p != 0.5
# ------------------------------------------------------------

def test_randomness(bits):

    n = len(bits)
    number_of_ones = np.sum(bits)

    p_hat = number_of_ones / n

    # Standard error assuming p = 0.5
    standard_error = np.sqrt(0.25 / n)

    z = (p_hat - 0.5) / standard_error

    # Two-sided p-value
    p_value = 2 * norm.sf(abs(z))

    alpha = 0.05

    if p_value < alpha:
        decision = "Reject H0: significant bias detected"
    else:
        decision = "Fail to reject H0: no significant bias detected"

    return p_hat, z, p_value, decision


# ------------------------------------------------------------
# Experiment
# ------------------------------------------------------------

shots = 100000

# Quantum source
quantum_bits = quantum_random_bits(shots)

# Biased classical source
classical_bits = classical_random_bits(shots)


# ------------------------------------------------------------
# Quantum result
# ------------------------------------------------------------

print("QUANTUM RANDOM SOURCE")

p, z, p_value, decision = test_randomness(
    quantum_bits
)

print("Number of bits:", shots)
print("Measured P(1):", p)
print("Z-score:", z)
print("p-value:", p_value)
print("Decision:", decision)


# ------------------------------------------------------------
# Classical result
# ------------------------------------------------------------

print("\nBIASED CLASSICAL SOURCE")

p, z, p_value, decision = test_randomness(
    classical_bits
)

print("Number of bits:", shots)
print("Measured P(1):", p)
print("Z-score:", z)
print("p-value:", p_value)
print("Decision:", decision)


QUANTUM RANDOM SOURCE
Number of bits: 100000
Measured P(1): 0.50195
Z-score: 1.2332882874656725
p-value: 0.21746822614354056
Decision: Fail to reject H0: no significant bias detected

BIASED CLASSICAL SOURCE
Number of bits: 100000
Measured P(1): 0.5091
Z-score: 5.755345341506448
p-value: 8.646485099844124e-09
Decision: Reject H0: significant bias detected
